In [1]:
import mysql.connector
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import joblib
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
def get_db_connection():
    return mysql.connector.connect(
        host='localhost',
        user='root',
        password='Vishal@74',
        database='chatbot_db'
    )

conn = get_db_connection()
cursor = conn.cursor(dictionary=True)

# Load intents & patterns
cursor.execute("SELECT id, tag FROM intents")
intents = cursor.fetchall()

# Load patterns
patterns = []
labels = []
for intent in intents:
    cursor.execute("SELECT pattern FROM patterns WHERE intent_id = %s", (intent['id'],))
    intent_patterns = cursor.fetchall()
    for p in intent_patterns:
        patterns.append(p['pattern'])
        labels.append(intent['tag'])

cursor.close()
conn.close()

print(f"Loaded {len(patterns)} patterns")

Loaded 8 patterns


In [ ]:
def get_db_connection():
    return mysql.connector.connect(
        host='localhost',
        user='root',
        password='Vishal@74',
        database='chatbot_db'
    )

conn = get_db_connection()
cursor = conn.cursor(dictionary=True)

# Load intents & patterns with better query
cursor.execute("SELECT id, tag FROM intents ORDER BY id")
intents = cursor.fetchall()

# Load patterns with better data handling
patterns = []
labels = []
print("Loading patterns from database...")
for intent in intents:
    cursor.execute("SELECT pattern FROM patterns WHERE intent_id = %s", (intent['id'],))
    intent_patterns = cursor.fetchall()
    for p in intent_patterns:
        if p['pattern'] and p['pattern'].strip():  # Ensure non-empty patterns
            patterns.append(p['pattern'])
            labels.append(intent['tag'])
            print(f"  - {intent['tag']}: {p['pattern']}")

cursor.close()
conn.close()

print(f"\n✅ Loaded {len(patterns)} patterns from {len(intents)} intents")
print(f"Unique intents: {len(set(labels))}")
print(f"Intent distribution: {dict(zip(*np.unique(labels, return_counts=True)))}")

In [ ]:
# Enhanced text preprocessing
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Add custom Indian English stopwords
custom_stopwords = {'hai', 'hain', 'ho', 'ki', 'ko', 'ke', 'ka', 'bhi', 'mein', 'par', 'aur', 'lekin', 'kyunki'}
stop_words.update(custom_stopwords)

def clean_text(text):
    """
    Enhanced text cleaning:
    1. Lowercase
    2. Tokenize
    3. Remove non-alphanumeric
    4. Remove stopwords (English + Indian)
    5. Lemmatize
    """
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens 
              if word.isalnum() and word not in stop_words]
    return ' '.join(tokens)

# Test preprocessing
test_text = "Hello! Kya haal hai? I want to know about fees"
cleaned = clean_text(test_text)
print(f"Original: {test_text}")
print(f"Cleaned: {cleaned}")

# Apply preprocessing to all patterns
print(f"\nPreprocessing {len(patterns)} patterns...")
cleaned_patterns = [clean_text(p) for p in patterns]

# Remove empty patterns after cleaning
valid_data = [(p, l) for p, l in zip(cleaned_patterns, labels) if p.strip()]
cleaned_patterns = [p for p, l in valid_data]
labels = [l for p, l in valid_data]

print(f"✅ Preprocessed {len(cleaned_patterns)} valid patterns")
print(f"Removed {len(patterns) - len(cleaned_patterns)} empty patterns")

Vectorizer & Encoder saved!


In [ ]:
# Enhanced TF-IDF Vectorization with better parameters
vectorizer = TfidfVectorizer(
    max_features=1000,  # Increased for better coverage
    ngram_range=(1, 2),  # Include bigrams for better context
    min_df=1,
    max_df=0.95,
    sublinear_tf=True
)

X = vectorizer.fit_transform(cleaned_patterns).toarray()

# Enhanced label encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
y_cat = to_categorical(y)  # One-hot encoding

print(f"✅ Vectorized data shape: {X.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Number of intent classes: {len(label_encoder.classes_)}")
print(f"Intent classes: {label_encoder.classes_.tolist()}")

# Save vectorizer and encoder
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
print("✅ Vectorizer & Encoder saved!")

Epoch 1/50


C:\Users\ACER\OneDrive\Desktop\Project\rule_based_chatbot\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1667 - loss: 1.1387


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.1667 - loss: 1.1387 - val_accuracy: 0.5000 - val_loss: 1.1009


Epoch 2/50


...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.8333 - loss: 0.3984 - val_accuracy: 0.5000 - val_loss: 1.1926


Epoch 50/50



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 1.0000 - loss: 0.4717


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - accuracy: 1.0000 - loss: 0.4717 - val_accuracy: 0.5000 - val_loss: 1.1972


Model trained and saved!

In [ ]:
# Enhanced Neural Network Model
from tensorflow.keras.layers import Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

# Split data for better validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

# Build improved model
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # Proper input layer
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(y_train.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Enhanced Model Architecture:")
model.summary()

# Train with early stopping and better monitoring
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

print("🚀 Enhanced Training Started...")
print(f"Training on {X_train.shape[0]} samples")
print(f"Validating on {X_val.shape[0]} samples")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=8,
    callbacks=[early_stopping],
    verbose=1
)

# Save the trained model
model.save('intent_model.h5')
print("✅ Model saved as 'intent_model.h5'")

# Show final results
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\n🎯 Final Results:")
print(f"Training Accuracy: {final_train_acc:.4f}")
print(f"Validation Accuracy: {final_val_acc:.4f}")
print(f"Best Validation Accuracy: {max(history.history['val_accuracy']):.4f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step


(np.str_('Address'), np.float32(0.42811644))
